In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch


In [2]:
torch.version.cuda

'12.1'

In [3]:
import sys
import json

import argparse
import random

import numpy as np
import torch.nn as nn
import torch.nn.functional as F


from feat_func import data_process
from models import DualGatedSage
from utils import DGraphFin
from utils.evaluator import Evaluator
from utils.utils import prepare_folder
from importlib import reload
import torch_geometric.utils
from torch_geometric.utils import k_hop_subgraph

from torch_geometric.nn.models.tgn import TGNMemory
from torch_geometric.nn.models import GAT

from torch_geometric.transforms import FeaturePropagation

from time import time, sleep

# choose the GPU
DEVICE = "1"

In [4]:
import torch
import gc

def reset_training_env(vars_to_clear: dict):
    """
    Membersihkan model, optimizer, dan loss_fn dari GPU memory.
    
    Parameters:
        vars_to_clear (dict): kamus variabel seperti {'model': model, 'optimizer': optimizer, 'loss_fn': loss_fn}
    """
    for name, var in vars_to_clear.items():
        try:
            del var
            print(f"{name} deleted.")
        except Exception as e:
            print(f"Gagal hapus {name}: {e}")

    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

In [5]:
class MaskedMSELoss(nn.Module):
    def __init__(self):
        super(MaskedMSELoss, self).__init__()

    def forward(self, input, target, mask):
        mask = mask.float()
        loss = (input - target) ** 2
        loss = loss * mask
        denom = (mask.sum() + 1e-8)
        if denom == 0:
            return torch.tensor(0.0, device=input.device, requires_grad=True)
        return loss.sum() / denom

def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train(model, data, optimizer, data_imputation,num_layers, loss_fn, undersampling=True, gate_attr_idx=[], engineered_feat_idx=dict()):
    model.train()

    optimizer.zero_grad()
    if undersampling:
        neg_idx = data.train_neg[
            torch.randperm(data.train_neg.size(0))[: data.train_pos.size(0)]
        ]
        train_idx = torch.cat([data.train_pos, neg_idx], dim=0)
    
        nodeandneighbor, edge_index, node_map, mask = k_hop_subgraph(
            train_idx, num_layers, data.edge_index, relabel_nodes=True, num_nodes=data.x.size(0)
        ) 
        gate_attr = data.x[nodeandneighbor][:, :0]
        if  gate_attr_idx:
            gate_attr = torch.cat([data.x[nodeandneighbor][:, start:end] for key in gate_attr_idx 
                          for (start, end) in [engineered_feat_idx[key]]], dim=1)
        
        # untuk gear sage normal dengan under sampling
        out, x_feat, missing_mask = model(
            data.x[nodeandneighbor],
            edge_index,
            data.edge_attr[mask],
            data.edge_timestamp[mask],
            data.edge_direct[mask],
            gate_attr
        )

        neg_idx_sum = neg_idx.sum()

        # class_loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        if loss_fn == "bce_loss":
            # Ambil output logits (tanpa sigmoid), lalu squeeze untuk ubah shape [N, 1] → [N]
            logits = out[node_map].squeeze()
            # logits = torch.clamp(logits, min=-100, max=100)
            
            # Pastikan label-nya bertipe float dan cocok dimensinya
            targets = data.y[train_idx].float()
            
            # Hitung loss
            device = f"cuda:{DEVICE}" if torch.cuda.is_available() else "cpu"
            pos_weight = torch.tensor([1], dtype=torch.float32).to(device)
            class_loss = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)
        else:
            class_loss = F.nll_loss(out[node_map], data.y[train_idx])
        
        if data_imputation:
            encoder_criterion = MaskedMSELoss()
            loss_encoder = encoder_criterion(x_feat, data.x[nodeandneighbor][:,:17], 1.0-missing_mask)
            # loss_encoder = encoder_criterion(x_feat, data.x[nodeandneighbor], 1.0-missing_mask)
        
            total_loss = class_loss + loss_encoder
        else:
            total_loss = class_loss

    else:
        gate_attr = data.x[:, :0]
        if  gate_attr_idx:
            gate_attr = torch.cat([data.x[:, start:end] for key in gate_attr_idx 
                          for (start, end) in [engineered_feat_idx[key]]], dim=1)
        neg_idx_sum = 0
        out = model(
            data.x,
            data.edge_index,
            data.edge_attr,
            data.edge_timestamp,
            data.edge_direct,
            gate_attr
        )

        # class_loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        if loss_fn == "bce_loss":
            # Ambil output logits (tanpa sigmoid), lalu squeeze untuk ubah shape [N, 1] → [N]
            logits = out[data.train_mask].squeeze()
            # logits = torch.clamp(logits, min=-100, max=100)
            
            # Pastikan label-nya bertipe float dan cocok dimensinya
            targets = data.y[data.train_mask].float()
            
            # Hitung loss
            device = f"cuda:{DEVICE}" if torch.cuda.is_available() else "cpu"
            pos_weight = torch.tensor([1], dtype=torch.float32).to(device)
            class_loss = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)
        else:
            class_loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        
        if data_imputation:
            encoder_criterion = MaskedMSELoss()
            loss_encoder = encoder_criterion(x_feat, data.x[data.train_mask][:,:17], 1.0-missing_mask)
            # loss_encoder = encoder_criterion(x_feat, data.x[nodeandneighbor], 1.0-missing_mask)
        
            total_loss = class_loss + loss_encoder
        else:
            total_loss = class_loss
    
    total_loss.backward()

    nn.utils.clip_grad_norm_(model.parameters(), 2.0)

    optimizer.step()
    torch.cuda.empty_cache()
    return total_loss.item(), neg_idx_sum

@torch.no_grad()
def test(model, data, gate_attr_idx=[], engineered_feat_idx=dict()):
    model.eval()
    #  gear sage biasa
    gate_attr = data.x[:, :0]
    if  gate_attr_idx:
        gate_attr = torch.cat([data.x[:, start:end] for key in gate_attr_idx 
                          for (start, end) in [engineered_feat_idx[key]]], dim=1)
    out, _, _ = model(
        data.x, data.edge_index, data.edge_attr, data.edge_timestamp, data.edge_direct, gate_attr
    )
    y_pred = out.exp()
    
    val_loss = F.nll_loss(out[data.valid_mask], data.y[data.valid_mask])
    
    return y_pred, val_loss, model.embeddings


def main(hyper_parameters):
    BASE_SEED = 1405
    
    parser = argparse.ArgumentParser(description="DualGatedSage for DGraphFin Dataset")
    parser.add_argument("--dataset", type=str, default="DGraphFin")
    parser.add_argument("--model", type=str, default=hyper_parameters["model"])
    parser.add_argument("--device", type=int, default=DEVICE)
    parser.add_argument("--epochs", type=int, default=hyper_parameters["epochs"]) #500
    parser.add_argument("--hiddens", type=int, default=hyper_parameters["hiddens"]) #96
    parser.add_argument("--layers", type=int, default=hyper_parameters["num_layers"])
    parser.add_argument("--dropout", type=float, default=hyper_parameters["dropout"])
    parser.add_argument("-f", type=str, default="")

    args = parser.parse_args()
    print(args)

    device = f"cuda:{args.device}" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)
    model_dir = prepare_folder(hyper_parameters["experiment_name"], hyper_parameters["model"])
    print("model_dir:", model_dir)
   
    valid_auc_runs = []
    test_auc_runs = []
    runs = []
    
    x_start = f"Experiment: {hyper_parameters.get('experiment_name')} started"
    
    for run in range(hyper_parameters["runs"]):
        set_seed(BASE_SEED+run)    
        dataset = DGraphFin(root="./dataset", name="DGraphFin", preprocess=hyper_parameters["preprocess"])
        nlabels = 2
        data = dataset[0]
    
        split_idx = {
            "train": data.train_mask,
            "valid": data.valid_mask,
            "test": data.test_mask,
        }
    
        data, engineered_feat_idx = data_process(data, hyper_parameters["preprocess"], hyper_parameters["feature_flag"], hyper_parameters["data_imputation"], hyper_parameters["time_density"])
        data = data.to(device)

        # hitung panjang dimensi gate
        gate_attr_channels = sum([end-start for key in hyper_parameters["gate_attr_idx"]
                for (start, end) in [engineered_feat_idx[key]]])
        
        y_train, y_valid = data.y[data.train_mask], data.y[data.valid_mask]
        train_idx = split_idx["train"].to(device)
        
        data.train_pos = train_idx[data.y[train_idx] == 1]
        data.train_neg = train_idx[data.y[train_idx] == 0]
        out_channels = args.hiddens if hyper_parameters["loss_fn"] == "bce_loss" else nlabels
        model = DualGatedSage(
            in_channels=data.x.size(-1),
            hidden_channels=args.hiddens,
            out_channels=out_channels,
            edge_attr_channels=hyper_parameters["edge_attr_channels"], #50
            time_channels=hyper_parameters["time_channels"], #50
            num_layers=args.layers,
            dropout=args.dropout,
            activation=hyper_parameters["activation"],
            bn=hyper_parameters["bn"],
            time_encoding=hyper_parameters["time_encoding"],
            data_imputation=hyper_parameters["data_imputation"],
            edge_gated=hyper_parameters["edge_gated"],
            residual_gated=hyper_parameters["residual_gated"],
            intra_layer_gated=hyper_parameters["intra_layer_gated"],
            concat=hyper_parameters["concat"],
            heads=hyper_parameters["heads"],
            loss_fn=hyper_parameters["loss_fn"],
            conv=hyper_parameters["conv"],
            gate_mechanism=hyper_parameters["gate_mechanism"],
            gate_attr_channels=gate_attr_channels
        ).to(device)

        model_parameters = sum(p.numel() for p in model.parameters())
    
        print(f"Model {args.model} initialized")
        if hyper_parameters["optimizer"] == "adam":
            optimizer = torch.optim.Adam(model.parameters(), lr=hyper_parameters["lr"], weight_decay=hyper_parameters["weight_decay"])
        elif hyper_parameters["optimizer"] == "adamw":
            optimizer = torch.optim.AdamW(model.parameters(), lr=hyper_parameters["lr"], weight_decay=hyper_parameters["weight_decay"])
        else:
            raise Exception("Optimizer unknown")
        evaluator = Evaluator("auc")
        
        best_auc = 0.0
        epochs_no_improve = 0
        patience = 100  # jumlah epoch tanpa peningkatan sebelum berhenti
        early_stop = False
        final_embeddings = None
        for epoch in range(1, args.epochs + 1):
            set_seed(((BASE_SEED+run)*1000) + epoch)
            if hyper_parameters["early_stop"] and early_stop:
                print(f"Early stopping at epoch {epoch}")
                break
            start = time()
            loss, neg_idx_sum = train(model, data, optimizer, hyper_parameters["data_imputation"], hyper_parameters["num_layers"], hyper_parameters["loss_fn"], hyper_parameters["undersampling"], hyper_parameters["gate_attr_idx"], engineered_feat_idx)
            train_time = time()
            out, val_loss, embeddings = test(model, data, hyper_parameters["gate_attr_idx"], engineered_feat_idx)
            preds_train, preds_valid = out[data.train_mask], out[data.valid_mask]
            test_time = time()
            if torch.isinf(preds_train).any() or torch.isnan(preds_train).any():
                print("Warning: y_true contains inf or NaN values!")
            
            train_auc = evaluator.eval(y_train, preds_train)["auc"]
            valid_auc = evaluator.eval(y_valid, preds_valid)["auc"]
    
            if valid_auc >= best_auc:
                epochs_no_improve = 0
                best_auc = valid_auc
                torch.save(model.state_dict(), model_dir + f"{run+1:02d}-model.pt")
                preds = out[data.test_mask].cpu().numpy()
                final_embeddings = embeddings
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    early_stop = True

            pred_temp = out[data.test_mask].cpu().numpy()
            test_temp = evaluator.eval(data.y[data.test_mask], pred_temp)["auc"]
            if epoch % 10 == 0:
                print(f"Run: {run+1:02d} Epoch: {epoch:02d}, Neg Idx Sum: {neg_idx_sum}, Params: {model_parameters}, Train Loss: {loss:.4f}, Val Loss: {val_loss:.4f} Train: {train_auc:.2%} Valid: {valid_auc:.2%} Test: {test_temp:.2%} Best: {best_auc:.4%}")
            runs.append({"run":run+1,"epoch":epoch, "loss": loss, "val_loss": float(val_loss), "train": train_auc, "valid": valid_auc,
                         "best": best_auc, "train_time":train_time-start, "test_time":test_time-train_time, "neg_idx_sum":int(neg_idx_sum), "test_temp": test_temp})
    
        
        test_auc = evaluator.eval(data.y[data.test_mask], preds)["auc"]
        print(f"\n\nTest AUC Run {run+1:02d}: {test_auc}")
        
        run_result = f"Run: {run+1:02d} Epoch: {epoch:02d}, Loss: {loss:.4f} Train: {train_auc:.2%} Valid: {valid_auc:.2%} Best: {best_auc:.4%} Test: {test_auc:.2%}"
        
        test_auc_runs.append(test_auc)
        valid_auc_runs.append(best_auc)
        reset_training_env(dict(model=model, optimizer=optimizer))

    print(f"Best valid auc: {np.mean(valid_auc_runs)}±{np.std(valid_auc_runs)}", flush=True)
    print(f"Best test auc: {np.mean(test_auc_runs)}±{np.std(test_auc_runs)}", flush=True)
    x_result = f"Experiment: {hyper_parameters.get('experiment_name')} Best valid auc: {np.mean(valid_auc_runs)}±{np.std(valid_auc_runs)} Best test auc: {np.mean(test_auc_runs)}±{np.std(test_auc_runs)}"

    artifact = dict(
        hyper_parameters=hyper_parameters,
        num_of_params=model_parameters,
        runs=runs,
        valid_auc_runs=valid_auc_runs,
        test_auc_runs=test_auc_runs,
        best_valid=f"{np.mean(valid_auc_runs)}±{np.std(valid_auc_runs)}",
        best_test=f"{np.mean(test_auc_runs)}±{np.std(test_auc_runs)}",
    )

    with open(f"{model_dir}/artifact.json", "w") as f:
        json.dump(artifact, f, indent=4)

    


In [ ]:
hyper_parameters = [
    dict(
        experiment_name="eks-223",
        epochs=500,
        hiddens=40,
        dropout=0.3,
        edge_gated=True,
        residual_gated=True,
        intra_layer_gated=False,
        heads=1,
        concat=False,
        activation="elu",
        bn=True,
        preprocess=True,
        runs=10,
        model="Edge-gatedSAGE",
        edge_attr_channels=10, #50
        time_channels=20, #50,
        weight_decay=7e-5,
        lr=1e-2,
        num_layers=3,
        time_encoding="deterministic", # time_encoder / temporal_encoding / deterministic
        data_imputation=False,
        feature_flag=True,
        time_density=False,
        early_stop=True,
        loss_fn="nll_loss",
        undersampling=True,
        conv="edge_gated", #bikernel, edge_gated
        optimizer="adam",
        gate_mechanism="glu",
        gate_attr_idx=[] # degree_feature, node_simililarity, missing_feature_flag, label_counts, label_feature, node_time_density
    )]

for h in hyper_parameters:
    main(h)